## 1. Load the tools and credit data
Each row is a credit case. `status` is the target: **0 = good credit**, **1 = bad credit**. Logistic regression will estimate the probability of class 1. Keep the CSV in the notebook’s working folder.

In [ ]:
import warnings
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sn
%matplotlib inline
from sklearn import metrics


In [ ]:
warnings.filterwarnings('ignore')
credit_df = pd.read_csv( "German Credit Data.csv" )
credit_df.info()

## 2. Explore the inputs and target
Preview different column ranges to make the many features easier to inspect. Features include account and credit history categories, loan amount, and age.

In [ ]:
credit_df.iloc[0:5,1:7]

In [ ]:
credit_df.iloc[0:5,7:]

Count good and bad credit cases. A majority class can make accuracy misleading, so pay attention to how many bad cases the model detects.

In [ ]:
credit_df['status'].value_counts()
# 0 - Good credit
# 1 - Bad credit

## 3. Choose and encode inputs
Create the list of input features and remove `status` so the answer is not included among the predictors.

In [ ]:
# Create a list of features to train our classification model.
X_features = list( credit_df.columns )
X_features.remove( 'status' )
X_features

One-hot encode categorical features. `drop_first=True` omits one indicator from each category to use as a reference level.

In [ ]:
encoded_credit_df = pd.get_dummies( credit_df[X_features],drop_first = True )

Inspect the names of the encoded columns. Each indicator is 0 or 1, representing whether a category applies.

In [ ]:
list(encoded_credit_df.columns)

Use the checking-account indicators as a concrete example: one account category is the reference category because its dummy column was dropped.

In [ ]:
encoded_credit_df[['checkin_acc_A12',
'checkin_acc_A13',
'checkin_acc_A14']].head(5)

## 4. Set up logistic regression
`Y` is the actual credit status. `X` contains the encoded inputs plus a constant, which represents the intercept in the statsmodels model.

In [ ]:
Y = credit_df.status
X = sm.add_constant( encoded_credit_df )

Keep 70% for training and 30% for testing. The fixed random state reproduces the same split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,Y,test_size = 0.3,random_state = 42)

## 5. Fit the first model
`sm.Logit` estimates the probability of **bad credit (1)** from the training rows. The fitted parameters are coefficients, not probabilities; positive coefficients increase the log odds of class 1, holding other inputs fixed.

In [ ]:
logit = sm.Logit(y_train, X_train)
logit_model = logit.fit()
logit_model.params

Inspect the model summary: coefficient signs, uncertainty, and p-values. A small p-value offers evidence of association under the model’s assumptions; it does not prove causation.

In [ ]:
logit_model.summary2()

## 6. Select variables and refit
This helper collects coefficients with p-values at most 0.05. This is a demonstration of model-based selection, not a guarantee of better predictive performance.

In [ ]:
def get_significant_vars( lm ):
    var_p_vals_df = pd.DataFrame( lm.pvalues )
    var_p_vals_df['vars'] = var_p_vals_df.index
    var_p_vals_df.columns = ['pvals', 'vars']
    return list( var_p_vals_df[var_p_vals_df.pvals <= 0.05]['vars'] )

Display the selected names. Note that the intercept (`const`) may also appear because the helper does not exclude it.

In [ ]:
significant_vars = get_significant_vars( logit_model )
significant_vars

Refit the model using selected columns. **Run-time caution:** if `significant_vars` includes `const`, `X_train[significant_vars]` already has a constant; the existing `sm.add_constant` call may be redundant. The original code is retained.

In [ ]:
# Now build the model again using only the significant variables.
final_logit = sm.Logit( y_train,sm.add_constant( X_train[significant_vars] ) ).fit()


Compare the smaller model’s summary with the original summary. Focus on signs and stability as well as p-values.

In [ ]:
final_logit.summary2()

## 7. Predict probabilities on held-out cases
Predict the probability of bad credit for each test row and put it beside its actual status. The existing constant-column caveat also applies to this call.

In [ ]:
y_pred = final_logit.predict( sm.add_constant( X_test[significant_vars] ) )

# Predicting on test data

y_pred_df = pd.DataFrame( { "actual": y_test,"predicted_prob": y_pred } )

Look at a sample of actual labels and predicted probabilities. A predicted probability such as 0.70 means an estimated 70% chance of class 1 under this model.

In [ ]:
# We can print the predictions of few test samples randomly using the sample method of DataFrame.
y_pred_df.sample(10, random_state = 42)

Convert probabilities to predicted labels: above **0.5 → bad credit (1)**, otherwise **good credit (0)**. This threshold is a decision choice, not a property of logistic regression.

In [ ]:
y_pred_df['predicted'] = y_pred_df.predicted_prob.map(lambda x: 1 if x > 0.5 else 0)

y_pred_df.sample(30, random_state = 42)

Inspect the predictions table and its data types before evaluating results.

In [ ]:
y_pred_df.info()

## 8. Read the confusion matrix
Load plotting and metric tools at the top of this notebook. The next cells arrange class **1 (bad credit)** first and class **0 (good credit)** second.

In [ ]:
# Creating Confusion Matrix


Define a heatmap function. With bad credit as the positive class, a **false negative** is an actual bad-credit case predicted good; a **false positive** is an actual good-credit case predicted bad.

In [ ]:
def draw_cm( actual, predicted ):
## Cret
    cm = metrics.confusion_matrix( actual, predicted, [1,0] )
    sn.heatmap(cm, annot=True, fmt='.2f',
    xticklabels = ["Bad credit", "Good Credit"] ,
    yticklabels = ["Bad credit", "Good Credit"] )
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

Plot the matrix. Read rows as actual outcomes and columns as predicted outcomes; check both kinds of errors rather than only the total correct.

In [ ]:
draw_cm( y_pred_df.actual,
y_pred_df.predicted )

### Understand the four boxes: bad credit is the “positive” result

The heatmap puts **actual results in rows** and **predictions in columns**. Read it like this:

| Actual case | Predicted bad credit | Predicted good credit |
|---|---|---|
| **Bad credit** | **TP (true positive):** correctly spotted a risky case | **FN (false negative):** missed a risky case |
| **Good credit** | **FP (false positive):** wrongly flagged a good case | **TN (true negative):** correctly identified a good case |

“Positive” simply means **bad credit (1)** here. “True” means the prediction was correct; “false” means it was wrong. For example, if a bad-credit applicant is predicted good, that is an **FN**. A good-credit applicant predicted bad is an **FP**.

### What do precision, recall, and F1 tell us?

Imagine **100 applicants predicted as bad credit**: if 80 actually have bad credit, **precision = 80/100 = 80%**. Precision answers: *When the model flags someone as risky, how often is it right?* Formula: **TP / (TP + FP)**. Use it when unnecessarily flagging good applicants is costly.

Imagine **100 actual bad-credit applicants**: if the model catches 70, **recall = 70/100 = 70%**. Recall answers: *Of all the risky applicants, how many did we catch?* Formula: **TP / (TP + FN)**. Use it when missing risky applicants is costly. Recall is also called **sensitivity**.

**F1 score** combines precision and recall into one number: **2 × precision × recall / (precision + recall)**. With 80% precision and 70% recall, F1 is about **74.7%**. Use it when both false alarms and missed risky cases matter. A high F1 needs both precision and recall to be reasonably high.

**Reading the next classification report:** Look at the row for **class 1** to assess bad-credit detection. The class 0 row answers a different question about good-credit cases. These numbers depend on the model's predictions; the 80% and 70% above are examples, not its measured scores.

Display the report. In a risk example, discuss the operational impact of missed bad-credit cases alongside the cost of incorrectly flagging good cases.

In [ ]:
print( metrics.classification_report( y_pred_df.actual,
y_pred_df.predicted ) )

## 10. ROC curve and AUC
ROC traces true positive rate against false positive rate as the probability threshold changes. AUC summarizes ranking across thresholds; roughly 0.5 means random ranking and 1.0 means perfect ranking.

In [ ]:
def draw_roc( actual, probs ):
    fpr, \
    tpr, \
    thresholds = metrics.roc_curve( actual,
    probs,
    drop_intermediate = False )
    auc_score = metrics.roc_auc_score( actual, probs )
    plt.figure(figsize=(8, 6))
    plt.plot( fpr, tpr, label='ROC curve (area = %0.2f)' % auc_score )
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate or [1 - True Negative Rate]')
    plt.ylabel('True Positive Rate')
    plt.legend(loc="lower right")
    plt.show()
    return fpr, tpr, thresholds

Plot ROC using the probabilities of **bad credit (1)**. The fixed-threshold classification report and threshold-varying ROC AUC answer different evaluation questions.

In [ ]:
# The above model is good at identifying good credits but not very good at identifying bad credits.
# This is because recall is high and precision is low.
fpr, tpr, thresholds = draw_roc( y_pred_df.actual,y_pred_df.predicted_prob)

## Save the trained model as a pickle file

Pickle saves the fitted model so you can load it later without training it again. Run this cell **after** the model training cells. The saved model expects the selected columns in `significant_vars`, with the same encoding and constant-column setup used during training.

Only load pickle files from sources you trust.

In [ ]:
import pickle

with open("credit_classification_model.pkl", "wb") as file:
    pickle.dump(final_logit, file)

print("Saved model to credit_classification_model.pkl")